# Regularized Regression


## 1. Imports

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


## 2. Load Processed Dataset

In [2]:
df = pd.read_csv("../data/processed/member_analysis_ready.csv")

print(df.shape)
df.head()



(500, 16)


,member_id,age,gender,region,plan_type,chronic_condition_count,engagement_score,ed_visits,ip_admits,monthly_cost,awv_completed,engagement_group,age_group,high_cost_member,has_acute_utilization,chronic_burden_group
0,M00001,69,Female,Rural,DSNP,3,72.6,1,1,2634.13,0,Q4,65-79,True,1,Moderate
1,M00002,32,Female,Suburban,Medicare Advantage,0,71.5,0,1,1632.38,1,Q4,18-34,False,1,Low
2,M00003,89,Male,Suburban,Medicaid,2,32.6,0,0,978.36,1,Q1,80+,False,0,Moderate
3,M00004,78,Male,Suburban,Medicare Advantage,7,56.0,3,0,2761.83,1,Q3,65-79,True,1,High
4,M00005,38,Female,Urban,DSNP,2,42.2,0,1,2001.45,1,Q1,35-49,True,1,Moderate


## 3. Define Target and Features


In [3]:
target = "monthly_cost"
X = df.drop(columns=[target, "member_id"], errors = "ignore") # Drop target & member_id if it exists, but don't raise an error if it doesn't
y = df[target] # Define target variable

numeric_cols = X.select_dtypes(include = ["int64", "float64"]).columns.tolist() # Identify numeric columns
categorical_cols = X.select_dtypes(include = ["str", "category", "bool"]).columns.tolist() # Identify categorical columns (including bools as categorical)



## 4. Train Test Split

In [4]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 42) # Split data into training and testing sets (80% train, 20% test) with a fixed random state for reproducibility


## 5. Build Preprocessing

In [5]:
preprocessor = ColumnTransformer( 
    transformers = [
        ("num", StandardScaler(), numeric_cols), # Scale numeric features using StandardScaler (mean=0, std=1)
        ("cat", OneHotEncoder(drop = "first", handle_unknown = "ignore"), categorical_cols) # Encode categorical features using OneHotEncoder, dropping the first category to avoid multicollinearity and ignoring unknown categories in test data
    ]
)

## 6. Build three models

In [6]:
linear_model = Pipeline(steps = [
    ("preprocessor", preprocessor), # Apply preprocessing steps defined in the preprocessor
    ("regressor", LinearRegression()) # Fit a Linear Regression model
])

ridge_model = Pipeline(steps = [
    ("preprocessor", preprocessor), # Apply preprocessing steps defined in the preprocessor
    ("regressor", Ridge(alpha = 1.0)) # Fit a Ridge Regression model with alpha=1.0 (L2 regularization)
])

Lasso_model = Pipeline(steps = [
    ("preprocessor", preprocessor), # Apply preprocessing steps defined in the preprocessor
    ("regressor", Lasso(alpha = 0.1, max_iter = 10000))
])

## 7. Fit all models

In [7]:
linear_model.fit(X_train, y_train) # Fit the Linear Regression model on the training data
ridge_model.fit(X_train, y_train) # Fit the Ridge Regression model on the training data
Lasso_model.fit(X_train, y_train) # Fit the Lasso Regression model on the training data



,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessor', ...), ('regressor', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformer

## 8. Evaluate all models

In [8]:
def evaluate_model(model, X_test, y_test, model_name):
    y_pred = model.predict(X_test) # Generate predictions on the test set 
    
    mae = mean_absolute_error(y_test, y_pred) # Calculate Mean Absolute Error
    rmse = np.sqrt(mean_squared_error(y_test, y_pred)) # Calculate Root Mean Squared Error
    r2 = r2_score(y_test, y_pred) # Calculate R-squared score

    return pd.DataFrame({
        "model" : [model_name], # Name of the model being evaluated
        "MAE" : [mae], # Mean Absolute Error
        "RMSE" : [rmse], # Root Mean Squared Error
        "R2" : [r2] # R-squared score
    })

results_df = pd.concat([
    evaluate_model(linear_model, X_test, y_test, "Linear Regression"), # Evaluate Linear Regression model and add results to the results DataFrame
    evaluate_model(ridge_model, X_test, y_test, "Ridge Regression"), # Evaluate Ridge Regression model and add results to the results DataFrame
    evaluate_model(Lasso_model, X_test, y_test, "Lasso Regression") # Evaluate Lasso Regression model and add results to the results DataFrame
], ignore_index = True) # Concatenate results into a single DataFrame and reset index


results_df 

,model,MAE,RMSE,R2
0,Linear Regression,87.344233,109.680259,0.974834
1,Ridge Regression,88.035524,110.276446,0.974560
2,Lasso Regression,87.668419,109.948212,0.974711


## 9.Extract coefficient for comparison 

In [9]:
feature_names = linear_model.named_steps["preprocessor"].get_feature_names_out() # Get the names of the features after preprocessing (including one-hot encoded features)
linear_coefs = linear_model.named_steps["regressor"].coef_ # Get the coefficients from the Linear Regression model
ridge_coefs = ridge_model.named_steps["regressor"].coef_ # Get the coefficients from the Ridge Regression model
lasso_coefs = Lasso_model.named_steps["regressor"].coef_ # Get the coefficients from the Lasso Regression model

coef_comparison = pd.DataFrame ({
    "feature" : feature_names,
    "linear_coef" : linear_coefs,
    "ridge_coef" : ridge_coefs,
    "lasso_coef" : lasso_coefs
})

coef_comparison["linear_coef_abs"] = coef_comparison["linear_coef"].abs() # Calculate absolute values of Linear Regression coefficients for easier comparison
coef_comparison["ridge_coef_abs"] = coef_comparison["ridge_coef"].abs() # Calculate absolute values of Ridge Regression coefficients for easier comparison
coef_comparison["lasso_coef_abs"] = coef_comparison["lasso_coef"].abs() # Calculate absolute values of Lasso Regression coefficients for easier comparison  

coef_comparison.head(20)

,feature,linear_coef,ridge_coef,lasso_coef,linear_coef_abs,ridge_coef_abs,lasso_coef_abs
0,num__age,71.805523,72.372283,75.840380,71.805523,72.372283,75.840380
1,num__chronic_condition_count,268.119156,263.241509,264.948340,268.119156,263.241509,264.948340
2,num__engagement_score,-13.801060,-11.354050,-10.333789,13.801060,11.354050,10.333789
3,num__ed_visits,191.972017,190.926272,192.401469,191.972017,190.926272,192.401469
4,num__ip_admits,332.204456,330.045437,332.263565,332.204456,330.045437,332.263565
5,num__awv_completed,-2.490399,-2.287983,-2.315499,2.490399,2.287983,2.315499
6,num__has_acute_utilization,12.568717,13.580378,12.209949,12.568717,13.580378,12.209949
7,cat__gender_Male,0.709996,1.086432,0.600014,0.709996,1.086432,0.600014
8,cat__region_Suburban,-2.973783,-2.029456,-1.507849,2.973783,2.029456,1.507849
9,cat__region_Urban,-24.179940,-24.092788,-23.144931,24.179940,24.092788,23.144931


## 11. Compare shrinkage visually or in a table

In [10]:
coef_comparison.sort_values("linear_coef_abs", ascending = False).head(20) # Sort features by absolute value of Linear Regression coefficients to identify the most influential features according to the Linear Regression model

coef_comparison["ridge_shrinkage"] = np.abs(coef_comparison["linear_coef"] - np.abs(coef_comparison["ridge_coef"])) # Calculate the shrinkage of coefficients from Linear to Ridge Regression to see how much Ridge is shrinking the coefficients compared to Linear Regression
coef_comparison["lasso_shrinkage"] = np.abs(coef_comparison["linear_coef"] - np.abs(coef_comparison["lasso_coef"])) # Calculate the shrinkage of coefficients from Linear to Lasso Regression to see how much Lasso is shrinking the coefficients compared to Linear Regression

coef_comparison.sort_values("lasso_shrinkage", ascending = False).head(20) # Sort features by the amount of shrinkage in Lasso Regression to identify which features are being most heavily penalized by Lasso's L1 regularization

,feature,linear_coef,ridge_coef,lasso_coef,linear_coef_abs,ridge_coef_abs,lasso_coef_abs,ridge_shrinkage,lasso_shrinkage
10,cat__plan_type_Medicaid,-377.055494,-364.511662,-375.390891,377.055494,364.511662,375.390891,741.567157,752.446386
11,cat__plan_type_Medicare Advantage,-221.517045,-209.344626,-219.592335,221.517045,209.344626,219.592335,430.861671,441.109380
9,cat__region_Urban,-24.179940,-24.092788,-23.144931,24.179940,24.092788,23.144931,48.272728,47.324871
2,num__engagement_score,-13.801060,-11.354050,-10.333789,13.801060,11.354050,10.333789,25.155109,24.134849
12,cat__engagement_group_Q2,-7.549659,-10.607354,-10.864042,7.549659,10.607354,10.864042,18.157013,18.413701
18,cat__age_group_80+,13.019918,10.818040,0.000000,13.019918,10.818040,0.000000,2.201878,13.019918
20,cat__chronic_burden_group_Low,15.751253,3.963423,5.770192,15.751253,3.963423,5.770192,11.787830,9.981061
14,cat__engagement_group_Q4,38.043716,30.115048,28.070576,38.043716,30.115048,28.070576,7.928668,9.973140
16,cat__age_group_50-64,14.616761,13.100043,7.314332,14.616761,13.100043,7.314332,1.516717,7.302428
13,cat__engagement_group_Q3,17.341483,12.206697,10.733742,17.341483,12.206697,10.733742,5.134786,6.607741


## Findings

1. Ridge and Lasso were introduced to control coefficient size and reduce overfitting pressure relative to baseline linear regression.
2. Ridge shrank coefficients without removing most predictors, making the model potentially more stable under correlated features.
3. Lasso applied stronger sparsity pressure and may have reduced some coefficients to exactly zero.
4. If Ridge or Lasso maintained similar test performance, that suggests the baseline model may have had some redundancy among predictors
5. Regularized coefficients are often more stable, but they are still conditional model-based associations, not casual effects.